# 00. Adathalmaz Áttekintés és Minőségi Riport

**TDK Kutatási Téma**: Budapest Főváros X. kerület (Kőbánya) lakóingatlan-piacának komplex térökonometriai, hedonikus és gépi tanulásos vizsgálata.

---

### 📖 Miről szól ez a kutatás, és miért éppen Kőbánya?
Budapest X. kerülete, **Kőbánya** a főváros egyik legizgalmasabb, legsokszínűbb területe. 
- Egyrészt magán viseli a **történelmi ipari és vasúti múlt** terheit (rozsdaövezetek, vágányhálózatok által elvágott területrészek, panelsorok).
- Másrészt hatalmas **fejlesztési potenciállal** bír (kitűnő kötöttpályás kapcsolatok, óriási zöldterületek, mint az Óhegy-park vagy a Népliget, illetve a tervezett Mázsa téri és rozsdaövezeti revitalizációk).

Kutatásunk célja, hogy **modern adattudományi, térinformatikai és ökonometriai eszközökkel** számszerűsítse a lakásárakat mozgató erőket: mennyit számít a vasút zaja, mennyit ér a metró közelsége, hol vannak a rejtett piaci lehetőségek, és hogyan nyerheti vissza a közösség a fejlesztések által generált értéknövekményt.

---

### 🗂️ Az adathalmaz háttere és megbízhatósága
Az elemzés gerincét az **ingatlan.com** kínálati adatbázisából kinyert és térinformatikailag (GIS) felbővített adatbázis adja. 
- **1 320 darab hirdetés** adatait dolgozzuk fel.
- Minden ingatlanhoz **169 különböző változó** tartozik: épületfizikai paraméterek (méret, szobaszám, állapot, építőanyag), hálózati gyalogos távolságok a tömegközlekedési csomópontoktól, környezeti terhelési adatok és intézményi ellátottsági mutatók.
- Az adatok szigorú tisztítási folyamaton mentek keresztül (duplikációk kiszűrése, hibás elgépelések javítása, statisztikai kiugró értékek kezelése).

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np

# Adathalmaz betöltése
df = load_szamitott_master()
print(f"Sikeresen betöltve: {len(df)} sor, {len(df.columns)} oszlop.")

Sikeresen betöltve: 1320 sor, 169 oszlop.


### 1. Alapvető Adatbázis KPI Mutatók (A piac mérete és szerkezete)

**📌 Mit látunk a fenti kártyákon?**
A vezérlőpult az adatbázis pontos, számított sarokszámait foglalja össze:
- **Összes hirdetés**: 1 320 db a kőbányai kínálatban.
- **Eladó lakások**: 1 140 db (a teljes piac **86.4%**-a).
- **Kiadó lakások**: 180 db (a piac **13.6%**-a).
- **Garantált pontos GIS koordinátával bíró lakások**: 296 db (**22.4%**).
- **Eladó lakások medián négyzetméterára**: 1 184 479 Ft/m² (~1.18 millió Ft/m²).
- **Átlagos alapterület**: 57.6 m² az eladó lakásoknál (és 52.5 m² a bérlakásoknál).

**🔍 Hogyan kell ezt értelmezni laikus szemmel?**
1. **Eladó vs. Kiadó arány**: A kínálat több mint nyolcszorosa eladó lakás (86.4%), és csak 13.6% kiadó albérlet. Ez tipikus magyar sajátosság, ahol a saját tulajdonú otthon dominál.
2. **Miért a mediánt nézzük az átlag helyett?** 
   - Ha van néhány 150-200 milliós luxuslakás, az átlagot erősen felfelé húzza (az eladó lakások átlagára 68.8 millió Ft). 
   - A **medián ár (65.0 millió Ft, illetve 1.18 millió Ft/m²)** az az érték, amelynél a lakások fele olcsóbb, fele pedig drágább. Ez adja a legőszintébb képet a piac derekáról.
3. **Garantált pontos GIS minta (N = 296 db)**: A hirdetések egy része csak utcát jelöl meg házszám nélkül. A mérnöki pontosságú távolságmérésekhez (pl. metró- vagy vasúttávolság) kizárólag a házszám szinten beazonosított 296 db ingatlant használjuk a térbeli modellekben.

**💡 Gyakorlati tanulság**: Kőbánya kínálati medián négyzetméterára ~1.18 millió Ft/m², ami versenyképes belépőt kínál a belső pesti kerületekhez képest.

In [2]:
n_total = len(df)
n_elado = len(df[df['listing_type'] == 'elado'])
n_kiado = len(df[df['listing_type'] == 'kiado'])
n_pontos = len(df[df['minta_garantalt_pontos'] == 1])
median_nm_ar = df[df['listing_type'] == 'elado']['nm_ar_huf'].median()
atlag_alapterulet = df['alapterulet_nm'].mean()

kpi_cards = [
    ("Összes Hirdetés", f"{n_total:,} db".replace(',', ' '), "Teljes adatbázis", "#1e3a8a"),
    ("Eladó Lakások", f"{n_elado:,} db".replace(',', ' '), f"{n_elado/n_total*100:.1f}% arány", "#2563eb"),
    ("Kiadó Lakások", f"{n_kiado:,} db".replace(',', ' '), f"{n_kiado/n_total*100:.1f}% arány", "#d97706"),
    ("Garantált Pontos GIS", f"{n_pontos:,} db".replace(',', ' '), f"{n_pontos/n_total*100:.1f}% koordinátás", "#16a34a"),
    ("Eladó Medián Ár/m²", fmt_huf(median_nm_ar), "Kínálati fajlagos ár", "#9333ea"),
    ("Átlagos Alapterület", f"{atlag_alapterulet:.1f} m²", "Kínálati átlag", "#0891b2"),
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. Kínálati Megoszlás Városrészenként (Hol van a legtöbb lakás?)

**📌 Mit látunk a grafikonon?**
Az oszlopdiagram Kőbánya különböző városrészeinek kínálati súlyát mutatja be, kékkel jelölve az eladó (1 140 db), sárgával a kiadó (180 db) lakásokat.

**🔍 Hogyan értelmezzük a látottakat?**
- **A piac súlypontjai**: A hirdetések túlnyomó része két nagy városrészben összpontosul: **Óhegyen** (amely a kerület legnagyobb lakó- és zöldövezete) és **Újhegyen** (a nagy kiterjedésű panel-lakótelepen).
- **A történelmi mag**: Kőbánya-Városközpont és Ligettelek stabil, vegyes (polgári tégla és panel) kínálatot nyújt.
- **Ipari és perifériás zónák**: Gyárdűlő, Laposdűlő és Felsőrákos alacsonyabb lakásszámmal szerepelnek, mivel ezek jelentős része ma is gazdasági, vasúti vagy alulhasznosított rozsdaövezeti funkciót tölt be.

**💡 Befektetői és önkormányzati szemüveg**:
A bérleti piac (sárga oszlopok) szinte kizárólag a jó közlekedésű Városközpontban, Újhegyen és Ligetteleken aktív. A külsőbb ipari területeken a lakások bérbeadási forgalma jóval alacsonyabb.

In [3]:
df_counts = df.groupby(['varosresz', 'listing_type']).size().reset_index(name='count')
df_counts['listing_nev'] = df_counts['listing_type'].map({'elado': 'Eladó', 'kiado': 'Kiadó'})

fig1 = px.bar(
    df_counts,
    x='varosresz',
    y='count',
    color='listing_nev',
    barmode='group',
    title='Hirdetések megoszlása városrészenként és típusonként',
    labels={'varosresz': 'Városrész', 'count': 'Hirdetések száma', 'listing_nev': 'Típus'},
    color_discrete_map={'Eladó': '#2563eb', 'Kiadó': '#f59e0b'},
    template=PLOTLY_TEMPLATE
)
fig1.update_layout(xaxis_tickangle=-30, height=450)
fig1.show()

### 3. Kutatási Változókatalógus és Teljes Adatszótár

A TDK kutatás 16 analitikai moduljában használt **összes kulcsváltozó rendszerezett leírása és kitöltöttségi vizsgálata**. A változók 6 funkcionális csoportba rendezve fedik le az ingatlanpiaci ármeghatározó tényezőket:
1. **Pénzügyi célváltozók**: vételárak, bérleti díjak és fajlagos árak (log transzformációval).
2. **Fizikai és épületjellemzők**: alapterület, szobaszám, panel/tégla szerkezet, műszaki állapot, lift, erkély és emelet.
3. **Hálózati séta távolságok**: a budapesti gyalogos úthálózaton (OSM nx) számított valós sétautak metróig, vasútállomásig, Mázsa térig, parkokig.
4. **Légvonalbeli környezeti hatások**: a vasúti pálya zaj- és rezgésemissziója, illetve referencia távolságok.
5. **Gyalogos izokrón dummyk**: 10-15 perces elérhetőségi zónák (750 m és 1125 m küszöbértékekkel).
6. **Térbeli GIS koordináták**: pontos geokódolt helymeghatározás (WGS84) és városrészi besorolás.

In [4]:
# 1. Teljes változó-metaadatbázis definiálása
valtozo_meta = [
    # Pénzügyi célváltozók
    ('price_huf', 'Kínálati vételár', 'Pénzügyi', 'HUF', 'Célváltozó (bruttó összeg)'),
    ('nm_ar_huf', 'Négyzetméterár', 'Pénzügyi', 'HUF/m²', 'Fajlagos célváltozó'),
    ('log_nm_ar', 'Fajlagos ár logaritmusa', 'Pénzügyi', 'ln(HUF/m²)', 'Ökonometriai regressziós cél'),
    ('ar_millio_ft', 'Vételár millió Ft-ban', 'Pénzügyi', 'M Ft', 'Leíró statisztikai mutató'),
    ('ar_ezer_ft_ho', 'Havi bérleti díj', 'Pénzügyi', 'ezer Ft/hó', 'Bérleti piac és Rent Gap'),
    # Fizikai és épületjellemzők
    ('alapterulet_nm', 'Alapterület', 'Fizikai', 'm²', 'Fizikai alaptulajdonság'),
    ('korrigalt_alapterulet_nm', 'Korrigált alapterület', 'Fizikai', 'm²', 'Fél erkéllyel súlyozva'),
    ('szobaszam_osszes', 'Összes szobaszám', 'Fizikai', 'db', 'Belső beosztási kontroll'),
    ('allapot', 'Műszaki állapot (szöveg)', 'Fizikai', 'kategória', 'Minőségi besorolás'),
    ('allapot_kod', 'Műszaki állapot index', 'Fizikai', '1-6 skála', 'Hedonikus minőségi rang'),
    ('is_panel', 'Panelszerkezet dummy', 'Fizikai', '0/1', 'Technológiai diszkont'),
    ('is_tegla', 'Tégla falazat dummy', 'Fizikai', '0/1', 'Hagyományos falazat'),
    ('has_lift', 'Lift megléte dummy', 'Fizikai', '0/1', 'Kényelmi felszereltség'),
    ('van_erkely', 'Erkély megléte dummy', 'Fizikai', '0/1', 'Kültéri kapcsolat prémiuma'),
    ('erkely_nm', 'Erkély mérete', 'Fizikai', 'm²', 'Kültéri felület nagysága'),
    ('emelet_szam', 'Emelet szintszám', 'Fizikai', 'szint', 'Függőleges elhelyezkedés'),
    ('epulet_kora_ev', 'Épület becsült kora', 'Fizikai', 'év', 'Amortizációs hatás'),
    ('epites_eve_kategoria', 'Építési korszak', 'Fizikai', 'korszak', 'Építészeti korcsoport'),
    # Hálózati közlekedési távolságok
    ('tavolsag_metro_halozati_m', 'Metróállomás hálózati táv.', 'Hálózati elérhetőség', 'méter', 'Gyalogos metróelérés (M2/M3)'),
    ('tavolsag_vasut_halozati_m', 'Vasútállomás hálózati táv.', 'Hálózati elérhetőség', 'méter', 'Állomási elérhetőség (TOD)'),
    ('tavolsag_mazsa_halozati_m', 'Mázsa tér hálózati táv.', 'Hálózati elérhetőség', 'méter', 'Városközpont & LVC fókusztáv'),
    ('tavolsag_villamos_halozati_m', 'Villamosmegálló hálózati táv.', 'Hálózati elérhetőség', 'méter', 'Felszíni kötöttpályás hálózat'),
    ('tavolsag_busz_halozati_m', 'Buszmegálló hálózati táv.', 'Hálózati elérhetőség', 'méter', 'Helyi buszhálózat elérése'),
    ('tavolsag_park_halozati_m', 'Park / zöldfelület hálózat', 'Hálózati elérhetőség', 'méter', 'Rekreációs zöldterület elérése'),
    ('tavolsag_belvaros_halozati_m', 'Belváros (Deák tér) hálózat', 'Hálózati elérhetőség', 'méter', 'Centrumtól való hálózati táv'),
    # Légvonalbeli környezeti externáliák
    ('tavolsag_vasut_m', 'Vasúti pálya légvonal (zaj)', 'Környezeti externália', 'méter', 'Immissziós zaj- és rezgésterhelés'),
    ('tavolsag_metro_m', 'Metróvonal légvonal', 'Környezeti externália', 'méter', 'Légvonalbeli referencia táv'),
    ('tavolsag_mazsa_m', 'Mázsa tér légvonal', 'Környezeti externália', 'méter', 'Légvonalbeli fókusztávolság'),
    ('tavolsag_belvaros_m', 'Belváros légvonal', 'Környezeti externália', 'méter', 'Légvonalbeli centrumtáv'),
    ('tavolsag_park_m', 'Legközelebbi park légvonal', 'Környezeti externália', 'méter', 'Zöldfelületi közelség'),
    # Gyalogos izokrón dummyk
    ('metro_10p_seta', 'Metró 10p séta (750m)', 'Gyalogos izokrón', '0/1', 'Metró vonzáskörzet'),
    ('vasut_10p_seta', 'Vasút 10p séta (750m)', 'Gyalogos izokrón', '0/1', 'Állomás gyalogos elérhetőség'),
    ('mazsa_10p_seta', 'Mázsa tér 10p séta', 'Gyalogos izokrón', '0/1', 'Központi akcióterület'),
    ('villamos_10p_seta', 'Villamos 10p séta', 'Gyalogos izokrón', '0/1', 'Villamos vonzáskörzet'),
    ('park_10p_seta', 'Park 10p séta', 'Gyalogos izokrón', '0/1', 'Közeli zöldterületi ellátás'),
    # Térbeli GIS koordináták
    ('geokodolt_lat', 'WGS84 Szélesség (Lat)', 'Térbeli GIS', 'fok', 'Térbeli pontkoordináta'),
    ('geokodolt_lon', 'WGS84 Hosszúság (Lon)', 'Térbeli GIS', 'fok', 'Térbeli pontkoordináta'),
    ('varosresz', 'Kőbányai Városrész', 'Térbeli GIS', 'név', 'Városrészi szegmens'),
    ('minta_garantalt_pontos', 'Pontos koordináta zászló', 'Térbeli GIS', '0/1', 'GIS szűrési jelző (N=296)')
]

# 2. Kitöltöttségi statisztikák számítása
stats_rows = []
dict_rows = []
for col, hu_name, kat, unit, role in valtozo_meta:
    if col in df.columns:
        n_valid = df[col].notna().sum()
        pct = (n_valid / len(df)) * 100
        stats_rows.append({
            'Technikai oszlop': col,
            'Változó megnevezése': hu_name,
            'Kategória': kat,
            'Kitöltöttség (%)': round(pct, 1),
            'Érvényes N': n_valid
        })
        dict_rows.append({
            'Kategória': kat,
            'Változó neve': hu_name,
            'Technikai név': f"<code>{col}</code>",
            'Mértékegység': unit,
            'Érvényes (db)': f"{n_valid:,} ({pct:.1f}%)".replace(',', ' '),
            'Szerepe a kutatásban': role
        })

df_qual = pd.DataFrame(stats_rows).sort_values(['Kategória', 'Kitöltöttség (%)'], ascending=[True, True])

# 3. Strukturált, kategóriánként színezett kitöltöttségi ábra
fig2 = px.bar(
    df_qual,
    x='Kitöltöttség (%)',
    y='Változó megnevezése',
    color='Kategória',
    orientation='h',
    title='Kutatási Változók Kitöltöttségi Aránya Funkcionális Kategóriánként (N = 1 320 db)',
    labels={'Kitöltöttség (%)': 'Kitöltöttség aránya (%)', 'Változó megnevezése': 'Változó'},
    range_x=[0, 105],
    template=PLOTLY_TEMPLATE,
    height=850
)
fig2.update_layout(yaxis=dict(tickfont=dict(size=10)), margin=dict(l=220, r=30, t=50, b=50))
fig2.show()

# 4. Teljes HTML Adatszótár megjelenítése
df_dict = pd.DataFrame(dict_rows)
html_dict = "<div style='overflow-x:auto; margin: 20px 0; max-height: 480px; overflow-y: auto;'>" + df_dict.to_html(classes='table table-bordered table-striped table-hover', index=False, escape=False) + "</div>"
display(HTML("<b>Részletes Kutatási Adatszótár és Változóleírás:</b>" + html_dict))

Kategória,Változó neve,Technikai név,Mértékegység,Érvényes (db),Szerepe a kutatásban
Pénzügyi,Kínálati vételár,price_huf,HUF,1 320 (100.0%),Célváltozó (bruttó összeg)
Pénzügyi,Négyzetméterár,nm_ar_huf,HUF/m²,1 320 (100.0%),Fajlagos célváltozó
Pénzügyi,Fajlagos ár logaritmusa,log_nm_ar,ln(HUF/m²),1 320 (100.0%),Ökonometriai regressziós cél
Pénzügyi,Vételár millió Ft-ban,ar_millio_ft,M Ft,1 140 (86.4%),Leíró statisztikai mutató
Pénzügyi,Havi bérleti díj,ar_ezer_ft_ho,ezer Ft/hó,180 (13.6%),Bérleti piac és Rent Gap
Fizikai,Alapterület,alapterulet_nm,m²,1 320 (100.0%),Fizikai alaptulajdonság
Fizikai,Korrigált alapterület,korrigalt_alapterulet_nm,m²,1 320 (100.0%),Fél erkéllyel súlyozva
Fizikai,Összes szobaszám,szobaszam_osszes,db,1 320 (100.0%),Belső beosztási kontroll
Fizikai,Műszaki állapot (szöveg),allapot,kategória,1 320 (100.0%),Minőségi besorolás
Fizikai,Műszaki állapot index,allapot_kod,1-6 skála,1 200 (90.9%),Hedonikus minőségi rang


### 4. Interaktív Adathalmaz Szűrő és Ellenőrző Pult

**📌 Mire szolgál ez az interaktív vezérlőpult?**
Az alábbi vezérlőkkel dinamikus szűréseket hajthat végre az adathalmazon:
- Kiválaszthatja, hogy eladó (1 140 db), kiadó (180 db), vagy mindkét (1 320 db) típusú hirdetést szeretné vizsgálni.
- Leszűrheti az adatbázist kizárólag a pontos GPS-koordinátával rendelkező ingatlanokra (N = 296 db).
- Kiválaszthat konkrét városrészeket (pl. csak Óhegy és Újhegy összehasonlítása).

**🔍 Mit lát az eredménytáblázatban?**
A szűrés után azonnal újraszámolódik a kiválasztott minta darabszáma, a szűrt átlagár, a medián négyzetméterár, és megjelenik a hirdetések valós előnézete a valós címekkel és árakkal.

**💡 Próbálja ki bátran!** Váltson át a „kiadó” opcióra, és figyelje meg, hogyan változik meg a mintaméret és az árszínvonal!

In [5]:
# Valós idejű kliensoldali adathalmaz-szűrő és feltáró vezérlőpult
import json

summary_rows = []
for (ltype, vr, pt), group in df.groupby(['listing_type', 'varosresz', 'minta_garantalt_pontos']):
    summary_rows.append({
        'type': str(ltype),
        'vr': str(vr),
        'pt': int(pt),
        'n': int(len(group)),
        'p_sum': float(group['price_huf'].dropna().sum()),
        'p_cnt': int(group['price_huf'].dropna().count()),
        'nm_sum': float(group['nm_ar_huf'].dropna().sum()),
        'nm_cnt': int(group['nm_ar_huf'].dropna().count()),
        'area_sum': float(group['alapterulet_nm'].dropna().sum()),
        'area_cnt': int(group['alapterulet_nm'].dropna().count())
    })

vr_list = sorted(list(df['varosresz'].dropna().unique()))

html_nb00 = f'''
<div id="nb00_filter_app" style="background:#ffffff; border:1px solid #cbd5e1; border-radius:12px; padding:22px; box-shadow:0 4px 6px -1px rgba(0,0,0,0.06); margin:18px 0; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;">
  <div style="display:flex; justify-content:space-between; align-items:center; border-bottom:2px solid #f1f5f9; padding-bottom:12px; margin-bottom:18px;">
    <div>
      <h3 style="margin:0; color:#1e3a8a; font-size:19px; font-weight:700;">🔎 Interaktív Adathalmaz Szűrő & Mintavételi Vezérlőpult</h3>
      <p style="margin:3px 0 0 0; color:#64748b; font-size:13px;">Dinamikus szegmentáció és statisztikai összegzés a teljes kőbányai ingatlanpiaci mintán</p>
    </div>
    <span style="background:#dbeafe; color:#1d4ed8; font-size:11px; font-weight:700; padding:4px 10px; border-radius:9999px;">Kliensoldali JS Motor</span>
  </div>

  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(260px, 1fr)); gap:18px; margin-bottom:18px;">
    <!-- Típus és Pontosság szűrő -->
    <div style="background:#f8fafc; padding:14px; border-radius:8px; border:1px solid #e2e8f0;">
      <div style="font-weight:700; color:#334155; margin-bottom:10px; font-size:13px; text-transform:uppercase;">1. Hirdetés Típusa:</div>
      <div style="display:flex; gap:12px; margin-bottom:14px;">
        <label style="font-size:13px; font-weight:600; color:#334155; cursor:pointer;">
          <input type="radio" name="nb00_type" value="all" checked onchange="filterNB00()"> Mindkettő
        </label>
        <label style="font-size:13px; font-weight:600; color:#2563eb; cursor:pointer;">
          <input type="radio" name="nb00_type" value="elado" onchange="filterNB00()"> Eladó lakások
        </label>
        <label style="font-size:13px; font-weight:600; color:#d97706; cursor:pointer;">
          <input type="radio" name="nb00_type" value="kiado" onchange="filterNB00()"> Kiadó lakások
        </label>
      </div>

      <div style="font-weight:700; color:#334155; margin-bottom:8px; font-size:13px; text-transform:uppercase;">2. Térbeli Pontosság:</div>
      <label style="display:flex; align-items:center; gap:8px; font-size:13px; font-weight:600; color:#059669; cursor:pointer;">
        <input type="checkbox" id="nb00_exact" onchange="filterNB00()"> Csak garantált pontos koordinátás (N=296)
      </label>
    </div>

    <!-- Városrészek szűrő -->
    <div style="background:#f8fafc; padding:14px; border-radius:8px; border:1px solid #e2e8f0;">
      <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:8px;">
        <div style="font-weight:700; color:#334155; font-size:13px; text-transform:uppercase;">3. Városrészek Kijelölése:</div>
        <div style="font-size:11px;">
          <a href="javascript:void(0)" onclick="toggleAllVR(true)" style="color:#2563eb; margin-right:8px; text-decoration:none; font-weight:600;">Mind</a>
          <a href="javascript:void(0)" onclick="toggleAllVR(false)" style="color:#dc2626; text-decoration:none; font-weight:600;">Töröl</a>
        </div>
      </div>
      <div id="nb00_vr_chips" style="display:flex; flex-wrap:wrap; gap:6px; max-height:110px; overflow-y:auto; padding:4px;">
        {"".join([f'<label style="font-size:11px; background:#ffffff; border:1px solid #cbd5e1; border-radius:14px; padding:3px 8px; cursor:pointer;"><input type="checkbox" class="nb00_vr_cb" value="{vr}" checked onchange="filterNB00()"> {vr}</label>' for vr in vr_list])}
      </div>
    </div>
  </div>

  <!-- KPI Rács -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap:12px; margin-bottom:16px;">
    <div style="background:#eff6ff; border:1px solid #93c5fd; border-radius:10px; padding:12px 16px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#1e40af; text-transform:uppercase;">Szűrt Elemek</div>
      <div id="nb00_res_count" style="font-size:24px; font-weight:800; color:#1d4ed8; margin:3px 0;">-- db</div>
      <div style="font-size:11px; color:#2563eb;">Kiválasztott mintaelemszám</div>
    </div>

    <div style="background:#f0fdf4; border:1px solid #86efac; border-radius:10px; padding:12px 16px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#166534; text-transform:uppercase;">Átlagos Kínálati Ár</div>
      <div id="nb00_res_price" style="font-size:24px; font-weight:800; color:#15803d; margin:3px 0;">-- M Ft</div>
      <div id="nb00_lbl_price_type" style="font-size:11px; color:#16a34a;">Vételár átlag</div>
    </div>

    <div style="background:#faf5ff; border:1px solid #d8b4fe; border-radius:10px; padding:12px 16px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#6b21a8; text-transform:uppercase;">Fajlagos Négyzetméterár</div>
      <div id="nb00_res_nm" style="font-size:24px; font-weight:800; color:#7e22ce; margin:3px 0;">-- Ft/m²</div>
      <div style="font-size:11px; color:#9333ea;">Szűrt minta átlaga</div>
    </div>

    <div style="background:#fffbeb; border:1px solid #fde68a; border-radius:10px; padding:12px 16px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#92400e; text-transform:uppercase;">Átlagos Alapterület</div>
      <div id="nb00_res_area" style="font-size:24px; font-weight:800; color:#b45309; margin:3px 0;">-- m²</div>
      <div style="font-size:11px; color:#d97706;">Fizikai lakásméret</div>
    </div>
  </div>

  <!-- Városrészi Összesítő Mini Táblázat -->
  <div style="overflow-x:auto;">
    <table id="nb00_stat_table" class="table table-bordered table-striped table-hover" style="width:100%; font-size:12px; margin-top:8px;">
      <thead>
        <tr style="background:#f1f5f9; color:#334155;">
          <th>Városrész</th>
          <th style="text-align:right;">Hirdetésszám</th>
          <th style="text-align:right;">Átlagár (M Ft)</th>
          <th style="text-align:right;">Átlag Ár/m²</th>
          <th style="text-align:right;">Átlag Terület</th>
        </tr>
      </thead>
      <tbody></tbody>
    </table>
  </div>
</div>

<script>
(function() {{
  const data = {json.dumps(summary_rows)};

  window.toggleAllVR = function(state) {{
    document.querySelectorAll('.nb00_vr_cb').forEach(cb => cb.checked = state);
    filter();
  }};

  function filter() {{
    const type = document.querySelector('input[name="nb00_type"]:checked').value;
    const exactOnly = document.getElementById('nb00_exact').checked;
    const selectedVR = new Set();
    document.querySelectorAll('.nb00_vr_cb:checked').forEach(cb => selectedVR.add(cb.value));

    let totalN = 0, sumP = 0, cntP = 0, sumNM = 0, cntNM = 0, sumArea = 0, cntArea = 0;
    let vrMap = {{}};

    for (let r of data) {{
      if (type !== 'all' && r.type !== type) continue;
      if (exactOnly && r.pt !== 1) continue;
      if (!selectedVR.has(r.vr)) continue;

      totalN += r.n;
      sumP += r.p_sum; cntP += r.p_cnt;
      sumNM += r.nm_sum; cntNM += r.nm_cnt;
      sumArea += r.area_sum; cntArea += r.area_cnt;

      if (!vrMap[r.vr]) {{
        vrMap[r.vr] = {{ n: 0, p_sum: 0, p_cnt: 0, nm_sum: 0, nm_cnt: 0, area_sum: 0, area_cnt: 0 }};
      }}
      vrMap[r.vr].n += r.n;
      vrMap[r.vr].p_sum += r.p_sum; vrMap[r.vr].p_cnt += r.p_cnt;
      vrMap[r.vr].nm_sum += r.nm_sum; vrMap[r.vr].nm_cnt += r.nm_cnt;
      vrMap[r.vr].area_sum += r.area_sum; vrMap[r.vr].area_cnt += r.area_cnt;
    }}

    const avgP = cntP > 0 ? (sumP / cntP) : 0;
    const avgNM = cntNM > 0 ? (sumNM / cntNM) : 0;
    const avgArea = cntArea > 0 ? (sumArea / cntArea) : 0;

    document.getElementById('nb00_res_count').innerText = totalN.toLocaleString('hu-HU') + ' db';
    if (type === 'kiado') {{
      document.getElementById('nb00_res_price').innerText = Math.round(avgP / 1000).toLocaleString('hu-HU') + ' ezer Ft/hó';
      document.getElementById('nb00_lbl_price_type').innerText = 'Havi bérleti díj átlag';
    }} else {{
      document.getElementById('nb00_res_price').innerText = (avgP / 1e6).toFixed(1) + ' M Ft';
      document.getElementById('nb00_lbl_price_type').innerText = 'Kínálati vételár átlag';
    }}
    document.getElementById('nb00_res_nm').innerText = Math.round(avgNM).toLocaleString('hu-HU') + ' Ft/m²';
    document.getElementById('nb00_res_area').innerText = avgArea.toFixed(1) + ' m²';

    // Táblázat renderelése
    const tbody = document.querySelector('#nb00_stat_table tbody');
    let rowsHtml = '';
    const vrSorted = Object.keys(vrMap).sort((a,b) => vrMap[b].n - vrMap[a].n);
    for (let vr of vrSorted) {{
      const v = vrMap[vr];
      const vp = v.p_cnt > 0 ? (v.p_sum / v.p_cnt) : 0;
      const vnm = v.nm_cnt > 0 ? (v.nm_sum / v.nm_cnt) : 0;
      const va = v.area_cnt > 0 ? (v.area_sum / v.area_cnt) : 0;
      const pStr = type === 'kiado' ? (Math.round(vp/1000).toLocaleString('hu-HU') + ' ezer') : (vp / 1e6).toFixed(1) + ' M';
      rowsHtml += `<tr>
        <td><b>${{vr}}</b></td>
        <td style="text-align:right;">${{v.n}} db</td>
        <td style="text-align:right;">${{pStr}}</td>
        <td style="text-align:right;">${{Math.round(vnm).toLocaleString('hu-HU')}} Ft</td>
        <td style="text-align:right;">${{va.toFixed(1)}} m²</td>
      </tr>`;
    }}
    tbody.innerHTML = rowsHtml || '<tr><td colspan="5" style="text-align:center; color:#94a3b8;">Nincs találat a kiválasztott szűrőkkel.</td></tr>';
  }}

  window.filterNB00 = filter;
  setTimeout(filter, 50);
}})();
</script>
'''
display(HTML(html_nb00))

Városrész,Hirdetésszám,Átlagár (M Ft),Átlag Ár/m²,Átlag Terület
